# Channel audit: M6 confirmatory run in resumable shards

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Arshia-HZ/mas-channel-audit/blob/main/notebooks/confirm.ipynb)

Runs **every arm** on the **first 600 confirmatory problems** (levels 3 to 5, in the frozen
hash order of the confirm split) with **2 replicates** for the primary model (Qwen3-1.7B,
temperature 0.7, 2048 tokens), in shards of about 50 problems. Arms: the seven preregistered
arms, plus `failure_cause_v2` (the stricter failure-message prompt). A (subject, level) group is
never split across shards, so the placebo arm finds its donors inside the chosen 600.

Each finished shard is validated and **uploaded to the private dataset**. Run all again after a
disconnect: finished shards are skipped without loading the model, an interrupted shard
continues, and a shard that finished but did not upload is only uploaded.

**This notebook never shows an accuracy, a gain or any contrast.** It prints only run quality
(truncation, context overflow, extraction failure, leakage rejection by arm), as the plan
requires. The analysis code is frozen first (M7) and run once on the finished data. Please do
not compute accuracies from the uploaded shards before then.

**Upload needs a Hugging Face token with write access** (secret `HF_TOKEN`). Roughly a dozen
shards, each about an hour and a half to two hours on a T4; `MAX_NEW_SHARDS` bounds a session.

## 0. Settings

In [ ]:
MODEL_CONFIG = "configs/models/qwen3-1.7b.yaml"
BACKEND = "vllm"
REPLICATES = 2
SHARD_SIZE = 50
N_PROBLEMS = 600  # the first 600 confirm problems in the frozen order
WITH_V2 = True  # also run failure_cause_v2
MAX_NEW_SHARDS = None  # e.g. 2 to stop this session after two new shards
UPLOAD = True  # push each finished shard to the private dataset
HF_REPO = "Arshia-HZ/mas-channel-audit-raw"
REF = "main"  # branch, tag or commit of the repository to run
REPO_URL = "https://github.com/Arshia-HZ/mas-channel-audit.git"
USE_DRIVE = True  # Colab only: keep shards in Google Drive

## 1. Environment

In [ ]:
import os
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = os.path.exists("/kaggle/working")

if IN_COLAB:
    REPO = "/content/mas-channel-audit"
    if USE_DRIVE:
        from google.colab import drive

        drive.mount("/content/drive")
        RUN_ROOT = "/content/drive/MyDrive/mas-channel-audit"
    else:
        RUN_ROOT = "/content/mas-channel-audit-runs"
elif IN_KAGGLE:
    REPO = "/kaggle/working/mas-channel-audit"
    RUN_ROOT = "/kaggle/working/runs"
else:  # a local checkout
    REPO = os.getcwd()
    if os.path.basename(REPO) == "notebooks":
        REPO = os.path.dirname(REPO)
    RUN_ROOT = os.environ.get("MCA_RUN_ROOT", os.path.join(REPO, "runs"))
os.makedirs(RUN_ROOT, exist_ok=True)


def sh(*cmd, cwd=None, check=True):
    """Run a command, show its output in the cell, raise if it fails."""
    proc = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)
    print(proc.stdout, end="")
    print(proc.stderr, end="", file=sys.stderr)
    if check and proc.returncode:
        raise RuntimeError(f"{' '.join(cmd)} exited with {proc.returncode}")
    return proc


print("colab" if IN_COLAB else "kaggle" if IN_KAGGLE else "local", "| repo:", REPO)
print("results go to:", RUN_ROOT)

In [ ]:
if IN_COLAB or IN_KAGGLE:
    if not os.path.isdir(f"{REPO}/.git"):
        sh("git", "clone", "-q", REPO_URL, REPO)
    sh("git", "fetch", "-q", "--all", cwd=REPO)
    sh("git", "checkout", "-q", REF, cwd=REPO)
    if subprocess.run(["git", "symbolic-ref", "-q", "HEAD"], cwd=REPO).returncode == 0:
        sh("git", "pull", "-q", "--ff-only", cwd=REPO)  # on a branch: take the latest
    extra = "vllm" if BACKEND == "vllm" else "hf"
    sh(sys.executable, "-m", "pip", "install", "-q", "-e", f".[{extra},data]", cwd=REPO)

os.chdir(REPO)
sys.path.insert(
    0, f"{REPO}/src"
)  # a running kernel does not see a fresh editable install
sh("git", "log", "--oneline", "-1")

**Hugging Face token.** Needed to upload shards (write access) and to restore them from the
dataset. Store it as a Colab or Kaggle secret named `HF_TOKEN`; it is never printed.

In [ ]:
token = os.environ.get("HF_TOKEN")
if not token and IN_COLAB:
    from google.colab import userdata

    try:
        token = userdata.get("HF_TOKEN")
    except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
        token = None
if not token and IN_KAGGLE:
    from kaggle_secrets import UserSecretsClient

    try:
        token = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:  # the secret is simply not set
        token = None
if token:
    os.environ["HF_TOKEN"] = token
print("HF token:", "found" if token else "not set (fine for the Qwen models)")

In [ ]:
import shutil
from importlib import metadata

if shutil.which("nvidia-smi"):
    sh("nvidia-smi", check=False)
for pkg in ("vllm", "torch", "transformers", "numpy", "math-verify", "pydantic"):
    try:
        print(f"{pkg:<14}{metadata.version(pkg)}")
    except metadata.PackageNotFoundError:
        print(f"{pkg:<14}not installed")

## 2. Model and confirmatory problems

In [ ]:
from pathlib import Path

from channel_audit.models import load_model_config
from channel_audit.prompt_freeze import require_frozen

for version in ("v1", "v2" if WITH_V2 else "v1"):
    require_frozen(Path(REPO) / "prompts" / version)  # the prompts must be frozen

model = load_model_config(f"{REPO}/{MODEL_CONFIG}")
if model.backend != BACKEND:
    model = model.model_copy(update={"backend": BACKEND})
slug = model.model_id.split("/")[-1].lower()
RUN = f"{RUN_ROOT}/confirm_{slug}_{BACKEND}"
PREFIX = f"confirm/{slug}"
print(model.model_id, "@", model.revision[:12], "|", model.backend, model.dtype)
print("max_tokens:", model.max_tokens, "| max_model_len:", model.max_model_len)
print("shards go to:", RUN, "| remote:", HF_REPO if UPLOAD else "(upload off)")

In [ ]:
from channel_audit.data import (
    MATH_REVISION,
    assign_splits,
    confirmatory_problems,
    load_frozen,
    load_math_test,
    write_split,
)

SPLIT_DIR = os.environ.get("MCA_SPLIT_DIR", f"{REPO}/data/problems")
# Rebuilds the split from the pinned MATH revision and checks it against the committed
# manifest; refuses to continue if anything differs.
rows, dropped = load_math_test()
manifest = write_split(
    assign_splits(rows), SPLIT_DIR, n_rows=len(rows) + len(dropped), dropped=dropped
)
problems = confirmatory_problems(SPLIT_DIR, N_PROBLEMS)
pilot_ids = {p.problem_id for p in load_frozen(SPLIT_DIR, "pilot")}
overlap = pilot_ids & {p.problem_id for p in problems}
if overlap:
    raise RuntimeError(f"{len(overlap)} confirmatory problems are also pilot problems")
print("MATH revision:", MATH_REVISION)
print(
    f"{len(problems)} confirmatory problems (levels "
    f"{sorted({p.level for p in problems})}), none from the pilot"
)

## 3. Run the shards

The model is loaded **once**, on the first shard that needs it. Each shard is validated before
the next one starts.

In [ ]:
from channel_audit.channels import ARMS, VARIANT_ARMS
from channel_audit.runner import backend_for_run, config_for_model
from channel_audit.shards import (
    LazyBackend,
    hf_downloader,
    hf_uploader,
    make_shards,
    run_shards,
)

cfg = config_for_model(model, replicates=REPLICATES, log_prompts=True)
uploader = restorer = None
if UPLOAD:
    token = os.environ.get("HF_TOKEN")
    if not token:
        raise RuntimeError("UPLOAD needs the HF_TOKEN secret (write access)")
    uploader = hf_uploader(HF_REPO, token)
    restorer = hf_downloader(HF_REPO, token)

n_shards = len(make_shards(problems, SHARD_SIZE))
print(f"{len(problems)} problems in {n_shards} shards of about {SHARD_SIZE}")
results = run_shards(
    RUN,
    cfg,
    problems,
    LazyBackend(lambda: backend_for_run(cfg, problems)),
    size=SHARD_SIZE,
    arms=(*ARMS, *VARIANT_ARMS) if WITH_V2 else ARMS,
    uploader=uploader,
    restorer=restorer,
    remote_prefix=PREFIX,
    max_new_shards=MAX_NEW_SHARDS,
)

In [ ]:
shard_dirs = sorted(
    d for d in Path(RUN).glob("shard_*") if (d / "SHARD_DONE.json").exists()
)
for r in results:
    print(
        f"{r.name}: {r.status}, {r.n_generations} generations, "
        f"G3 {'pass' if r.g3_ok else 'FAIL'}, uploaded {r.uploaded}"
    )
ALL_DONE = len(shard_dirs) == n_shards
print(
    f"{len(shard_dirs)} of {n_shards} shards finished",
    "- run all again to continue" if not ALL_DONE else "",
)

## 4. Run quality (no outcomes)

Truncation, context overflow, extraction failure and failure-message leakage, by arm, over the
finished shards. No accuracy or contrast is computed here.

In [ ]:
import json

from channel_audit.quality import format_quality_report, run_quality
from channel_audit.runner import open_run

if shard_dirs:
    ctxs = [open_run(d) for d in shard_dirs]
    quality = run_quality(ctxs)
    print(format_quality_report(quality))
    with open(f"{RUN}/quality_report.json", "w", encoding="utf-8") as fh:
        json.dump(quality, fh, indent=1, sort_keys=True)
    if ALL_DONE and IN_COLAB:
        from google.colab import files

        files.download(f"{RUN}/quality_report.json")

**Send back:** the printed quality report. The raw shards are already in the private dataset;
leave them untouched until the analysis code is frozen (M7).